# Temporal & heterogeneous graphs — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## Real graphs change over time and contain different kinds of nodes and edges
Temporal and heterogeneous graph learning keeps the relation type and timestamp instead of flattening them away. The examples compute snapshot changes, time decay, typed adjacency, meta-paths, and temporal negatives.

In [ ]:
# 1) temporal snapshot difference
A0=np.array([[0,1,0],[1,0,0],[0,0,0]]); A1=np.array([[0,1,1],[1,0,0],[1,0,0]]); Delta=A1-A0
plt.figure(figsize=(4,3)); plt.imshow(Delta,cmap='coolwarm'); plt.colorbar(); plt.title('new edge appears at t=1')
assert Delta[0,2]==1 and Delta.sum()==2

In [ ]:
# 2) exponential time decay weights recent edges more
ages=np.array([1.,3.,5.]); weights=np.exp(-0.5*ages)
plt.figure(figsize=(5,3)); plt.bar(['age1','age3','age5'],weights); plt.title('temporal decay weights')
assert weights[0]>weights[-1] and abs(weights[0]-0.60653066)<1e-6

In [ ]:
# 3) heterogeneous graph stores one adjacency per relation
UI=np.array([[1,0],[1,1],[0,1]],float); UU=np.array([[0,1,0],[1,0,0],[0,0,0]],float)
plt.figure(figsize=(6,3)); plt.subplot(1,2,1); plt.imshow(UI); plt.title('user-item'); plt.subplot(1,2,2); plt.imshow(UU); plt.title('user-user')
assert UI.shape==(3,2) and UU.shape==(3,3)

In [ ]:
# 4) meta-path U-I-U counts shared items between users
UI=np.array([[1,0],[1,1],[0,1]],float); M=UI@UI.T
plt.figure(figsize=(4,3)); plt.imshow(M,cmap='Blues'); plt.colorbar(); plt.title('U-I-U meta-path counts')
assert M[0,1]==1 and M[0,2]==0

In [ ]:
# 5) temporal negative sampling must respect time
edges=[(0,1,1),(0,2,3)]; candidate=(0,2,2); existed=any(u==candidate[0] and v==candidate[1] and t<=candidate[2] for u,v,t in edges)
plt.figure(figsize=(5,3)); plt.bar(['existed by t=2','exists after t=3'],[int(existed),1]); plt.title('time-aware negative')
assert existed is False